# Метод 4 — Set Transformer. Часть 3: оценка и итоги

## Пути

In [1]:
from pathlib import Path
import sys

ROOT = Path('/Users/ensamsanovich/auc_forecast')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

METHOD_DIR = ROOT / 'notebooks' / 'method_4_set_transformer'
FROZEN_DIR = METHOD_DIR / 'results_frozen'
OUT_DIR = ROOT / 'outputs' / 'method_4_set_transformer'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Frozen CSV:', FROZEN_DIR / 'ablation_summary_frozen.csv')

Frozen CSV: /Users/ensamsanovich/auc_forecast/notebooks/method_4_set_transformer/results_frozen/ablation_summary_frozen.csv


## Таблица ablation

In [2]:
import pandas as pd

summary = pd.read_csv(FROZEN_DIR / 'ablation_summary_frozen.csv')
summary = summary.sort_values('cv_smlar_mean').reset_index(drop=True)
summary

,variant,cv_smlar_mean,cv_smlar_std,monotone_violation,use_attention,use_distribution,loss,elapsed_min
0,A_full,27.532610,2.123637,0.000000,True,True,smlar_smooth,18.799350
1,B_no_attn,28.943006,2.296113,0.000000,False,True,smlar_smooth,1.807418
2,C_no_distr,34.284481,10.875610,0.000000,True,False,smlar_smooth,18.788654
3,E_none,61.255783,3.066996,0.003727,False,False,mse,1.824641
4,D_no_smlar,81.862755,13.878106,0.000000,True,True,mse,18.876781


In [3]:
folds = pd.read_csv(FROZEN_DIR / 'ablation_folds_frozen.csv')
folds

,variant,fold,smlar_flat,monotone_violation,source
0,A_full,0,27.53,0.0000,logs/ablation.log
1,A_full,1,27.53,0.0000,logs/ablation.log
2,A_full,2,27.53,0.0000,logs/ablation.log
3,A_full,3,27.53,0.0000,logs/ablation.log
4,A_full,4,27.53,0.0000,logs/ablation.log
5,B_no_attn,0,28.94,0.0000,logs/ablation.log
6,B_no_attn,1,28.94,0.0000,logs/ablation.log
7,B_no_attn,2,28.94,0.0000,logs/ablation.log
8,B_no_attn,3,28.94,0.0000,logs/ablation.log
9,B_no_attn,4,28.94,0.0000,logs/ablation.log


In [ ]:
import sys, time
from pathlib import Path

ROOT = Path('/Users/ensamsanovich/auc_forecast')
METHOD_DIR = ROOT / 'notebooks' / 'method_4_set_transformer'
RESULTS_DIR = METHOD_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for p in (ROOT, METHOD_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from core.runner import load_unified_data
from src.train import VARIANTS
from src.eval import evaluate_variant, fold_records_to_df

VARIANTS_TO_EVAL = ['A_full', 'B_no_attn', 'C_no_distr', 'D_no_smlar', 'E_none']

variant_map = dict(VARIANTS) 
data = load_unified_data()
print(
    f"train={len(data['split']['train_idx'])}  "
    f"holdout={len(data['split']['holdout_idx'])}  "
    f"folds={len(data['split']['folds'])}"
)

eval_results = []
all_fold_records = []
t_overall = time.time()
for vname in VARIANTS_TO_EVAL:
    if vname not in variant_map:
        raise ValueError(f"Unknown variant {vname!r}; known: {list(variant_map)}")
    print(f"\n=== {vname} ===")
    t0 = time.time()
    ev = evaluate_variant(
        variant_name=vname,
        variant_overrides=dict(variant_map[vname]),
        data=data,
        verbose=True,
        do_cv=True,
        do_holdout=True,
        checkpoint_dir=RESULTS_DIR,
    )
    elapsed = time.time() - t0
    cv = ev['cv']; ho = ev['holdout']; ci = ev['ci']
    print(
        f"  CV  flat={cv['smlar_flat_mean']:.2f}±{cv['smlar_flat_std']:.2f}  "
        f"y1={cv['smlar_per_target']['at_least_one']:.2f}  "
        f"y2={cv['smlar_per_target']['at_least_two']:.2f}  "
        f"y3={cv['smlar_per_target']['at_least_three']:.2f}  "
        f"mono={cv['monotone_viol_mean']*100:.2f}%"
    )
    print(
        f"  HO  flat={ho['smlar_flat']:.2f}  "
        f"y1={ho['smlar_per_target']['at_least_one']:.2f}  "
        f"y2={ho['smlar_per_target']['at_least_two']:.2f}  "
        f"y3={ho['smlar_per_target']['at_least_three']:.2f}  "
        f"mono={ho['monotone_viol']*100:.2f}%"
    )
    print(
        f"  CP  cov y1={ci['coverage_per_target']['at_least_one']:.3f}  "
        f"y2={ci['coverage_per_target']['at_least_two']:.3f}  "
        f"y3={ci['coverage_per_target']['at_least_three']:.3f}  "
        f"(α={ci['alpha']})"
    )
    print(f"  total elapsed: {elapsed/60:.1f} min")
    eval_results.append(ev)
    for rec in (ev.get('cv_fold_records') or []):
        all_fold_records.append({'variant': vname, **rec})
print(f"\nTotal {len(eval_results)} variants in {(time.time()-t_overall)/60:.1f} min")

train=806  holdout=202  folds=5

=== A_full ===
    ep   0  trainL=1.5521  valSMLAR=217.35%  best=217.35%
    ep   5  trainL=0.6033  valSMLAR=92.70%  best=82.62%
    ep  10  trainL=0.3675  valSMLAR=49.92%  best=49.92%
    ep  15  trainL=0.2958  valSMLAR=52.79%  best=49.82%
    ep  20  trainL=0.2471  valSMLAR=41.32%  best=41.32%
    ep  25  trainL=0.2308  valSMLAR=41.50%  best=40.54%
    ep  30  trainL=0.2099  valSMLAR=41.02%  best=36.66%
    ep  35  trainL=0.2024  valSMLAR=37.59%  best=36.66%
    ep  40  trainL=0.1952  valSMLAR=36.88%  best=36.66%
    ep  45  trainL=0.1862  valSMLAR=36.41%  best=35.95%
    ep  49  trainL=0.1868  valSMLAR=36.34%  best=35.95%
  [A_full] fold 0 best ep 44  SMLAR=35.95%  (166.8s)
    ep   0  trainL=1.4704  valSMLAR=230.61%  best=230.61%
    ep   5  trainL=0.5265  valSMLAR=64.80%  best=64.80%
    ep  10  trainL=0.4210  valSMLAR=81.52%  best=56.25%
    ep  15  trainL=0.3353  valSMLAR=50.71%  best=46.33%
    ep  20  trainL=0.3129  valSMLAR=43.29%  best=37.12%

In [ ]:
import pandas as pd
from core.reporting import save_method_results
from build_results import build_results_from_eval

OUT_DIR_LOCAL = ROOT / 'outputs' / 'method_4_set_transformer'
OUT_DIR_LOCAL.mkdir(parents=True, exist_ok=True)
PRIMARY = VARIANTS_TO_EVAL[0]
result = build_results_from_eval(eval_results, primary_variant=PRIMARY)
save_method_results('set_transformer', result)
print(
    f"Wrote results/per_method/set_transformer.{{json,md}}.\n"
    f"  primary = {PRIMARY}\n"
    f"  CV  {result['cv']['smlar_flat_mean']:.2f} ± {result['cv']['smlar_flat_std']:.2f}\n"
    f"  HO  {result['holdout']['smlar_flat']:.2f}\n"
    f"  CP  mean cov = "
    f"{sum(result['ci']['coverage_per_target'].values())/3:.3f}\n"
    f"  variants in extra: {len(result['extra']['variants'])}"
)
ablation_rows = []
for ev in eval_results:
    cv = ev['cv']; ho = ev['holdout']; ci = ev['ci']
    ablation_rows.append({
        'variant': ev['variant'],
        'cv_smlar_mean': cv['smlar_flat_mean'],
        'cv_smlar_std': cv['smlar_flat_std'],
        'cv_y1': cv['smlar_per_target']['at_least_one'],
        'cv_y2': cv['smlar_per_target']['at_least_two'],
        'cv_y3': cv['smlar_per_target']['at_least_three'],
        'cv_mono_pct': cv['monotone_viol_mean'] * 100,
        'ho_smlar': ho['smlar_flat'],
        'ho_y1': ho['smlar_per_target']['at_least_one'],
        'ho_y2': ho['smlar_per_target']['at_least_two'],
        'ho_y3': ho['smlar_per_target']['at_least_three'],
        'ho_mono_pct': ho['monotone_viol'] * 100,
        'cp_cov_y1': ci['coverage_per_target']['at_least_one'],
        'cp_cov_y2': ci['coverage_per_target']['at_least_two'],
        'cp_cov_y3': ci['coverage_per_target']['at_least_three'],
        'cp_width_y1': ci['width_per_target']['at_least_one'],
        'cp_width_y2': ci['width_per_target']['at_least_two'],
        'cp_width_y3': ci['width_per_target']['at_least_three'],
        'ho_best_epoch': ho.get('best_epoch'),
    })
ablation_df = pd.DataFrame(ablation_rows)
ablation_path = OUT_DIR_LOCAL / 'ablation_local.csv'
ablation_df.to_csv(ablation_path, index=False)
print(f"Wrote {ablation_path}")
ablation_df
primary_ev = next(ev for ev in eval_results if ev['variant'] == PRIMARY)
ho = primary_ev['holdout']
holdout_summary_df = pd.DataFrame([{
    'variant': PRIMARY,
    'holdout_smlar': ho['smlar_flat'],
    'smlar_y1': ho['smlar_per_target']['at_least_one'],
    'smlar_y2': ho['smlar_per_target']['at_least_two'],
    'smlar_y3': ho['smlar_per_target']['at_least_three'],
    'monotone_violation': ho['monotone_viol'],
    'best_epoch': ho.get('best_epoch'),
    'elapsed_s': ho.get('elapsed_s'),
}])
holdout_summary_path = RESULTS_DIR / 'holdout_summary.csv'
holdout_summary_df.to_csv(holdout_summary_path, index=False)
print(f"Wrote {holdout_summary_path}")
if all_fold_records:
    fold_df = pd.DataFrame(all_fold_records)
    fold_path = OUT_DIR_LOCAL / 'cv_folds_local.csv'
    fold_df.to_csv(fold_path, index=False)
    print(f"Wrote {fold_path}")

Wrote results/per_method/set_transformer.{json,md}.
  primary = A_full
  CV  31.66 ± 2.40
  HO  21.21
  CP  mean cov = 0.906
  variants in extra: 5
Wrote /Users/ensamsanovich/auc_forecast/outputs/method_4_set_transformer/ablation_local.csv
Wrote /Users/ensamsanovich/auc_forecast/notebooks/method_4_set_transformer/results/holdout_summary.csv
Wrote /Users/ensamsanovich/auc_forecast/outputs/method_4_set_transformer/cv_folds_local.csv


In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path('/Users/ensamsanovich/auc_forecast')
METHOD_DIR = ROOT / 'notebooks' / 'method_4_set_transformer'
FROZEN_DIR = METHOD_DIR / 'results_frozen'

summary = pd.read_csv(FROZEN_DIR / 'ablation_summary_frozen.csv')
print(summary.to_string())

      variant  cv_smlar_mean  cv_smlar_std  monotone_violation  use_attention  use_distribution          loss  elapsed_min
0      A_full      27.532610      2.123637            0.000000           True              True  smlar_smooth    18.799350
1   B_no_attn      28.943006      2.296113            0.000000          False              True  smlar_smooth     1.807418
2  C_no_distr      34.284481     10.875610            0.000000           True             False  smlar_smooth    18.788654
3  D_no_smlar      81.862755     13.878106            0.000000           True              True           mse    18.876781
4      E_none      61.255783      3.066996            0.003727          False             False           mse     1.824641
